In [2]:
!pip install pdfplumber
!pip install skillNer
!pip install sentence_transformers
!pip install transformers
!pip install spaCy
!pip install pyresparser

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 28.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 4.7 MB/s eta 0:00:00
  Created wheel for skillNer: filename=skillNer-1.0.3-py3-none-any.whl size=25625 sha256=6b72e30da6f44bade0cbe04187d562bf352426eee590a7cc772b7c221dd4afd6
  Stored in directory: /root/.cache/pip/wheels/05/61/2f/331f7cbd5980aceaec089ca82ce49c282523d8365bf99e8013
Successfully built skillNer
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [5]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
"""
AI-Powered Resume Screening System with Enhanced Modern Gradio UI
Enhanced with progress tracking and improved visibility
"""

import gradio as gr
import pdfplumber
import spacy
from skillNer.general_params import SKILL_DB
from skillNer.skill_extractor_class import SkillExtractor
from spacy.matcher import PhraseMatcher
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import pandas as pd
import numpy as np
from pathlib import Path
import re
from typing import List, Dict, Tuple
import warnings

warnings.filterwarnings('ignore')


class ResumeScreeningSystem:
    """End-to-end resume screening system with semantic matching and skill extraction"""

    def __init__(self):
        """Initialize all models and tools"""
        print("Initializing Resume Screening System...")
        self.nlp = spacy.load("en_core_web_lg")
        self.skill_extractor = SkillExtractor(self.nlp, SKILL_DB, PhraseMatcher)
        self.similarity_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=-1)
        print("System Ready!\n")

    def extract_text_from_pdf(self, pdf_path: str) -> str:
        """Extract clean text from PDF resume"""
        text = ""
        try:
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
            return text.strip()
        except Exception as e:
            print(f"Error extracting text from {pdf_path}: {e}")
            return ""

    def preprocess_text(self, text: str) -> Tuple[spacy.tokens.Doc, List[str]]:
        """Tokenize and preprocess text using spaCy"""
        doc = self.nlp(text)
        cleaned_tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_stop and not token.is_punct and token.is_alpha
        ]
        return doc, cleaned_tokens

    def extract_skills(self, text: str) -> Dict[str, List[str]]:
        """Extract skills using SkillNer"""
        try:
            annotations = self.skill_extractor.annotate(text)
            skills = []
            if 'results' in annotations:
                for result in annotations['results'].get('full_matches', []):
                    skill_name = result['doc_node_value']
                    if skill_name not in skills:
                        skills.append(skill_name)
                for result in annotations['results'].get('ngram_scored', []):
                    skill_name = result['doc_node_value']
                    if skill_name not in skills:
                        skills.append(skill_name)
            return {'skills': skills}
        except Exception as e:
            print(f"SkillNer extraction error: {e}")
            return {'skills': []}

    def extract_entities(self, doc: spacy.tokens.Doc) -> Dict[str, List[str]]:
        """Extract named entities"""
        entities = {'education': [], 'organizations': [], 'experience': []}

        for ent in doc.ents:
            if ent.label_ == "ORG":
                entities['organizations'].append(ent.text)
            elif ent.label_ in ["DATE", "TIME"]:
                entities['experience'].append(ent.text)

        education_patterns = [
            r'\b(B\.?S\.?|M\.?S\.?|Ph\.?D\.?|Bachelor|Master|Doctorate)\b',
            r'\b(Computer Science|Engineering|Business|MBA)\b'
        ]
        for pattern in education_patterns:
            matches = re.findall(pattern, doc.text, re.IGNORECASE)
            entities['education'].extend(matches)

        for key in entities:
            entities[key] = list(set(entities[key]))

        return entities

    def extract_all_features(self, text: str) -> Dict:
        """Complete feature extraction pipeline"""
        doc, tokens = self.preprocess_text(text)
        skills_data = self.extract_skills(text)
        entities_data = self.extract_entities(doc)

        return {
            'skills': skills_data['skills'],
            'education': entities_data['education'],
            'organizations': entities_data['organizations'],
            'experience': entities_data['experience'],
            'tokens': tokens,
            'full_text': text
        }

    def normalize_skills(self, skills: List[str]) -> List[str]:
        """Normalize and deduplicate skills"""
        normalized = [skill.lower().strip() for skill in skills]
        seen = set()
        unique_skills = []
        for skill in normalized:
            if skill not in seen:
                seen.add(skill)
                unique_skills.append(skill)
        return unique_skills

    def compute_similarity(self, resume_text: str, jd_text: str) -> float:
        """Compute semantic similarity"""
        resume_embedding = self.similarity_model.encode(resume_text, convert_to_tensor=True)
        jd_embedding = self.similarity_model.encode(jd_text, convert_to_tensor=True)
        similarity = np.dot(resume_embedding, jd_embedding) / (
                np.linalg.norm(resume_embedding) * np.linalg.norm(jd_embedding)
        )
        return float(similarity)

    def compute_skill_match(self, resume_skills: List[str], jd_skills: List[str]) -> Dict:
        """Compute skill-based matching metrics"""
        resume_skills_norm = set(self.normalize_skills(resume_skills))
        jd_skills_norm = set(self.normalize_skills(jd_skills))

        matched_skills = resume_skills_norm.intersection(jd_skills_norm)
        missing_skills = jd_skills_norm - resume_skills_norm

        match_percentage = (
            len(matched_skills) / len(jd_skills_norm) * 100
            if jd_skills_norm else 0
        )

        return {
            'matched_skills': list(matched_skills),
            'missing_skills': list(missing_skills),
            'match_percentage': match_percentage,
            'matched_count': len(matched_skills),
            'total_required': len(jd_skills_norm)
        }

    def summarize_resume(self, resume_text: str, max_length: int = 130, min_length: int = 30) -> str:
        """Generate concise summary of resume"""
        try:
            if len(resume_text.split()) > 1024:
                resume_text = ' '.join(resume_text.split()[:1024])
            summary = self.summarizer(resume_text, max_length=max_length, min_length=min_length, do_sample=False)
            return summary[0]['summary_text']
        except Exception as e:
            print(f"Summarization error: {e}")
            sentences = resume_text.split('.')[:3]
            return '. '.join(sentences) + '.'

    def screen_resume(self, resume_path: str, jd_text: str) -> Dict:
        """Complete resume screening pipeline"""
        resume_text = self.extract_text_from_pdf(resume_path)
        if not resume_text:
            return {'error': 'Failed to extract text from PDF'}

        resume_features = self.extract_all_features(resume_text)
        jd_features = self.extract_all_features(jd_text)

        semantic_similarity = self.compute_similarity(resume_text, jd_text)
        skill_match = self.compute_skill_match(resume_features['skills'], jd_features['skills'])
        summary = self.summarize_resume(resume_text)

        return {
            'candidate_name': Path(resume_path).stem,
            'semantic_similarity': round(semantic_similarity, 3),
            'skill_match_percentage': round(skill_match['match_percentage'], 2),
            'matched_skills': skill_match['matched_skills'],
            'missing_skills': skill_match['missing_skills'],
            'all_skills': resume_features['skills'],
            'education': resume_features['education'],
            'organizations': resume_features['organizations'],
            'summary': summary,
            'matched_count': skill_match['matched_count'],
            'total_required': skill_match['total_required'],
            'full_text': resume_text
        }


# Initialize the system
system = ResumeScreeningSystem()


def format_skills_section(skills: List[str], color: str, border_color: str, section_id: str) -> str:
    """Format skills section with show more/less functionality"""
    if not skills:
        return f"<p style='color: {color}; margin: 0; font-size: 16px;'>No skills found</p>"

    visible_skills = ' '.join([
        f"<span style='background: transparent; color: {color}; padding: 10px 20px; border-radius: 50px; font-size: 14px; font-weight: 600; border: 2px solid {border_color}; display: inline-block; margin: 4px;'>{skill}</span>"
        for skill in skills[:15]
    ])

    if len(skills) <= 15:
        return f"<div style='display: flex; flex-wrap: wrap; gap: 8px;'>{visible_skills}</div>"

    hidden_skills = ' '.join([
        f"<span style='background: transparent; color: {color}; padding: 10px 20px; border-radius: 50px; font-size: 14px; font-weight: 600; border: 2px solid {border_color}; display: inline-block; margin: 4px;'>{skill}</span>"
        for skill in skills[15:]
    ])

    return f"""
    <div style='display: flex; flex-wrap: wrap; gap: 8px;'>{visible_skills}</div>
    <div id='{section_id}-hidden' style='display: none; flex-wrap: wrap; gap: 8px; margin-top: 8px;'>{hidden_skills}</div>
    <button id='{section_id}-btn' onclick='
        var hidden = document.getElementById("{section_id}-hidden");
        var btn = document.getElementById("{section_id}-btn");
        if (hidden.style.display === "none") {{
            hidden.style.display = "flex";
            btn.textContent = "Show Less ▲";
        }} else {{
            hidden.style.display = "none";
            btn.textContent = "Show More ({len(skills) - 15}) ▼";
        }}
    ' style='background: linear-gradient(135deg, {color}, {border_color}); color: white; border: none; padding: 10px 24px; border-radius: 50px; font-weight: 600; cursor: pointer; font-size: 14px; margin-top: 12px; transition: all 0.3s ease;'>
        Show More ({len(skills) - 15}) ▼
    </button>
    """


def format_single_result(result: Dict) -> str:
    """Format single resume result as HTML with modern design"""
    if 'error' in result:
        return f"""
        <div style='background: linear-gradient(135deg, #ff6b6b 0%, #c92a2a 100%);
                    padding: 24px; border-radius: 16px; color: white;
                    box-shadow: 0 8px 32px rgba(201, 42, 42, 0.2);'>
            <h3 style='margin: 0; display: flex; align-items: center; gap: 12px;'>
                <span style='font-size: 28px;'>⚠️</span>
                {result['error']}
            </h3>
        </div>
        """

    composite = (result['semantic_similarity'] * 0.6 + result['skill_match_percentage'] / 100 * 0.4) * 100

    if composite >= 70:
        score_color = "#06b6d4"
        status_badge = "🌟 Excellent Match"
        status_bg = "#06b6d4"
    elif composite >= 50:
        score_color = "#f59e0b"
        status_badge = "⚡ Good Match"
        status_bg = "#f59e0b"
    else:
        score_color = "#ec4899"
        status_badge = "💡 Potential"
        status_bg = "#ec4899"

    doughnut_svg = f"""
    <svg viewBox="0 0 200 200" style="width: 200px; height: 200px;">
        <circle cx="100" cy="100" r="80" fill="none" stroke="#e5e7eb" stroke-width="25" transform="rotate(-90 100 100)"/>
        <circle cx="100" cy="100" r="80" fill="none" stroke="{score_color}" stroke-width="25"
                stroke-dasharray="{composite * 5.026} 502.6" stroke-linecap="round" transform="rotate(-90 100 100)"/>
        <text x="100" y="95" text-anchor="middle" font-size="42" font-weight="900" fill="#1f2937">
            {composite:.0f}%
        </text>
        <text x="100" y="115" text-anchor="middle" font-size="13" font-weight="700" fill="#6b7280" text-transform="uppercase" letter-spacing="1">
            Overall Match
        </text>
    </svg>
    """

    matched_section = format_skills_section(
        result['matched_skills'], '#059669', '#10b981', 'matched-skills'
    ) if result['matched_skills'] else "<p style='color: #065f46; margin: 0; font-size: 16px;'>No matching skills found</p>"

    missing_section = format_skills_section(
        result['missing_skills'], '#dc2626', '#ef4444', 'missing-skills'
    ) if result['missing_skills'] else "<p style='color: #991b1b; margin: 0; font-size: 16px;'>All required skills are present! 🎉</p>"

    all_skills_section = format_skills_section(
        result['all_skills'], '#7c3aed', '#8b5cf6', 'all-skills'
    ) if result['all_skills'] else "<p style='color: #5b21b6; margin: 0; font-size: 16px;'>No skills extracted</p>"

    education_section = format_skills_section(
        result['education'], '#d97706', '#f59e0b', 'education'
    ) if result['education'] else "<p style='color: #92400e; margin: 0; font-size: 16px;'>No education information found</p>"

    organization_section = format_skills_section(
        result['organizations'], '#0d9488', '#14b8a6', 'organizations'
    ) if result['organizations'] else "<p style='color: #134e4a; margin: 0; font-size: 16px;'>No organization information found</p>"

    html = f"""
    <div style='background: linear-gradient(to right, #1e1b4b 0%, #312e81 50%, #4c1d95 100%);
                padding: 40px; border-radius: 24px; color: white; margin-bottom: 24px;
                box-shadow: 0 20px 60px rgba(0, 0, 0, 0.3); min-height: 300px;'>
        <div style='display: grid; grid-template-columns: 1fr auto; gap: 40px; align-items: center;'>
            <div>
                <div style='display: inline-block; background: rgba(255, 255, 255, 0.1);
                            padding: 8px 20px; border-radius: 50px; margin-bottom: 16px;
                            backdrop-filter: blur(10px); border: 1px solid rgba(255, 255, 255, 0.2);'>
                    <span style='font-size: 14px; font-weight: 600; text-transform: uppercase; letter-spacing: 1px;'>
                        Candidate Profile
                    </span>
                </div>
                <h1 style='margin: 0 0 16px 0; font-size: 48px; font-weight: 800;
                           background: linear-gradient(to right, #fff, #e0e7ff);
                           -webkit-background-clip: text; -webkit-text-fill-color: transparent;
                           background-clip: text; word-break: break-word;'>
                    {result['candidate_name']}
                </h1>
                <div style='background: {status_bg}; padding: 12px 28px; border-radius: 50px;
                            font-weight: 700; font-size: 18px; box-shadow: 0 4px 16px rgba(0, 0, 0, 0.2);
                            display: inline-block;'>
                    {status_badge}
                </div>
            </div>
            <div style='display: flex; justify-content: center;'>
                {doughnut_svg}
            </div>
        </div>
    </div>

    <div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr)); gap: 16px; margin-bottom: 28px;'>
        <div style='background: white; padding: 24px; border-radius: 16px; box-shadow: 0 4px 20px rgba(0, 0, 0, 0.08);
                    border: 2px solid #e0e7ff; position: relative; overflow: hidden;'>
            <div style='position: absolute; top: 0; right: 0; width: 80px; height: 80px;
                        background: linear-gradient(135deg, #7c3aed, #a78bfa); opacity: 0.1; border-radius: 0 0 0 100%;'></div>
            <div style='position: relative; z-index: 1;'>
                <div style='display: flex; align-items: center; gap: 8px; margin-bottom: 10px;'>
                    <span style='font-size: 24px;'>🧠</span>
                    <h3 style='margin: 0; color: #1f2937; font-size: 13px; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px;'>Semantic</h3>
                </div>
                <p style='margin: 0; font-size: 36px; font-weight: 900; background: linear-gradient(135deg, #7c3aed, #a78bfa);
                          -webkit-background-clip: text; -webkit-text-fill-color: transparent;'>{result['semantic_similarity'] * 100:.1f}%</p>
            </div>
        </div>

        <div style='background: white; padding: 24px; border-radius: 16px; box-shadow: 0 4px 20px rgba(0, 0, 0, 0.08);
                    border: 2px solid #fef3c7; position: relative; overflow: hidden;'>
            <div style='position: absolute; top: 0; right: 0; width: 80px; height: 80px;
                        background: linear-gradient(135deg, #f59e0b, #fbbf24); opacity: 0.1; border-radius: 0 0 0 100%;'></div>
            <div style='position: relative; z-index: 1;'>
                <div style='display: flex; align-items: center; gap: 8px; margin-bottom: 10px;'>
                    <span style='font-size: 24px;'>⚡</span>
                    <h3 style='margin: 0; color: #1f2937; font-size: 13px; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px;'>Skills</h3>
                </div>
                <p style='margin: 0; font-size: 36px; font-weight: 900; background: linear-gradient(135deg, #f59e0b, #fbbf24);
                          -webkit-background-clip: text; -webkit-text-fill-color: transparent;'>{result['skill_match_percentage']:.1f}%</p>
            </div>
        </div>

        <div style='background: white; padding: 24px; border-radius: 16px; box-shadow: 0 4px 20px rgba(0, 0, 0, 0.08);
                    border: 2px solid #fce7f3; position: relative; overflow: hidden;'>
            <div style='position: absolute; top: 0; right: 0; width: 80px; height: 80px;
                        background: linear-gradient(135deg, #ec4899, #f472b6); opacity: 0.1; border-radius: 0 0 0 100%;'></div>
            <div style='position: relative; z-index: 1;'>
                <div style='display: flex; align-items: center; gap: 8px; margin-bottom: 10px;'>
                    <span style='font-size: 24px;'>✨</span>
                    <h3 style='margin: 0; color: #1f2937; font-size: 13px; font-weight: 700; text-transform: uppercase; letter-spacing: 0.5px;'>Matched</h3>
                </div>
                <p style='margin: 0; font-size: 36px; font-weight: 900; background: linear-gradient(135deg, #ec4899, #f472b6);
                          -webkit-background-clip: text; -webkit-text-fill-color: transparent;'>{result['matched_count']}<span style='font-size: 20px; opacity: 0.5;'>/{result['total_required']}</span></p>
            </div>
        </div>
    </div>

    <div style='background: linear-gradient(135deg, #dcfce7 0%, #bbf7d0 100%); padding: 32px; border-radius: 20px; margin-bottom: 24px;
                border: 2px solid #86efac; box-shadow: 0 8px 32px rgba(134, 239, 172, 0.2);'>
        <div style='display: flex; align-items: center; gap: 12px; margin-bottom: 20px;'>
            <div style='background: linear-gradient(135deg, #10b981, #059669); width: 48px; height: 48px; border-radius: 12px;
                        display: flex; align-items: center; justify-content: center; box-shadow: 0 4px 12px rgba(16, 185, 129, 0.3);'>
                <span style='font-size: 24px;'>✅</span>
            </div>
            <h2 style='margin: 0; color: #065f46; font-size: 24px; font-weight: 800;'>Matched Skills</h2>
        </div>
        {matched_section}
    </div>

    <div style='background: linear-gradient(135deg, #fef2f2 0%, #fecaca 100%); padding: 32px; border-radius: 20px; margin-bottom: 24px;
                border: 2px solid #fca5a5; box-shadow: 0 8px 32px rgba(252, 165, 165, 0.2);'>
        <div style='display: flex; align-items: center; gap: 12px; margin-bottom: 20px;'>
            <div style='background: linear-gradient(135deg, #ef4444, #dc2626); width: 48px; height: 48px; border-radius: 12px;
                        display: flex; align-items: center; justify-content: center; box-shadow: 0 4px 12px rgba(239, 68, 68, 0.3);'>
                <span style='font-size: 24px;'>⚠️</span>
            </div>
            <h2 style='margin: 0; color: #991b1b; font-size: 24px; font-weight: 800;'>Missing Skills</h2>
        </div>
        {missing_section}
    </div>

    <div style='background: linear-gradient(135deg, #ede9fe 0%, #ddd6fe 100%); padding: 32px; border-radius: 20px; margin-bottom: 24px;
                border: 2px solid #c4b5fd; box-shadow: 0 8px 32px rgba(196, 181, 253, 0.2);'>
        <div style='display: flex; align-items: center; gap: 12px; margin-bottom: 20px;'>
            <div style='background: linear-gradient(135deg, #8b5cf6, #7c3aed); width: 48px; height: 48px; border-radius: 12px;
                        display: flex; align-items: center; justify-content: center; box-shadow: 0 4px 12px rgba(139, 92, 246, 0.3);'>
                <span style='font-size: 24px;'>💼</span>
            </div>
            <h2 style='margin: 0; color: #5b21b6; font-size: 24px; font-weight: 800;'>All Candidate Skills</h2>
        </div>
        {all_skills_section}
    </div>

    <div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(300px, 1fr)); gap: 24px; margin-bottom: 24px;'>
        <div style='background: linear-gradient(135deg, #fef3c7 0%, #fde68a 100%); padding: 28px; border-radius: 20px; border: 2px solid #fcd34d;
                    box-shadow: 0 8px 32px rgba(252, 211, 77, 0.2);'>
            <div style='display: flex; align-items: center; gap: 12px; margin-bottom: 16px;'>
                <div style='background: linear-gradient(135deg, #f59e0b, #d97706); width: 40px; height: 40px; border-radius: 10px;
                            display: flex; align-items: center; justify-content: center; box-shadow: 0 4px 12px rgba(245, 158, 11, 0.3);'>
                    <span style='font-size: 20px;'>🎓</span>
                </div>
                <h2 style='margin: 0; color: #92400e; font-size: 20px; font-weight: 800;'>Education</h2>
            </div>
            {education_section}
        </div>

        <div style='background: linear-gradient(135deg, #ccfbf1 0%, #99f6e4 100%); padding: 28px; border-radius: 20px; border: 2px solid #5eead4;
                    box-shadow: 0 8px 32px rgba(94, 234, 212, 0.2);'>
            <div style='display: flex; align-items: center; gap: 12px; margin-bottom: 16px;'>
                <div style='background: linear-gradient(135deg, #14b8a6, #0d9488); width: 40px; height: 40px; border-radius: 10px;
                            display: flex; align-items: center; justify-content: center; box-shadow: 0 4px 12px rgba(20, 184, 166, 0.3);'>
                    <span style='font-size: 20px;'>🏢</span>
                </div>
                <h2 style='margin: 0; color: #134e4a; font-size: 20px; font-weight: 800;'>Organizations</h2>
            </div>
            {organization_section}
        </div>
    </div>

    <div style='background: linear-gradient(135deg, #f0f9ff 0%, #e0f2fe 100%); padding: 32px; border-radius: 20px; border: 2px solid #bae6fd;
                box-shadow: 0 8px 32px rgba(186, 230, 253, 0.3);'>
        <div style='display: flex; align-items: center; gap: 12px; margin-bottom: 20px;'>
            <div style='background: linear-gradient(135deg, #06b6d4, #0891b2); width: 48px; height: 48px; border-radius: 12px;
                        display: flex; align-items: center; justify-content: center; box-shadow: 0 4px 12px rgba(6, 182, 212, 0.3);'>
                <span style='font-size: 24px;'>📝</span>
            </div>
            <h2 style='margin: 0; color: #0c4a6e; font-size: 24px; font-weight: 800;'>Resume Summary</h2>
        </div>
        <p style='color: #0e7490; line-height: 1.8; margin: 0; font-size: 16px; font-weight: 500;'>{result['summary']}</p>
    </div>
    """
    return html


def format_multiple_results(results_df: pd.DataFrame) -> tuple:
    """Format multiple resume results as HTML and table"""
    table_html = f"""
    <div style='background: linear-gradient(to right, #1e1b4b 0%, #312e81 50%, #4c1d95 100%);
                padding: 40px; border-radius: 24px; color: white; margin-bottom: 32px;
                box-shadow: 0 20px 60px rgba(0, 0, 0, 0.3);'>
        <div style='display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 20px;'>
            <div>
                <div style='display: inline-block; background: rgba(255, 255, 255, 0.1);
                            padding: 8px 20px; border-radius: 50px; margin-bottom: 16px;
                            backdrop-filter: blur(10px); border: 1px solid rgba(255, 255, 255, 0.2);'>
                    <span style='font-size: 14px; font-weight: 600; text-transform: uppercase; letter-spacing: 1px;'>Batch Analysis</span>
                </div>
                <h1 style='margin: 0 0 8px 0; font-size: 42px; font-weight: 800;
                           background: linear-gradient(to right, #fff, #e0e7ff);
                           -webkit-background-clip: text; -webkit-text-fill-color: transparent;'>📊 Candidate Rankings</h1>
                <p style='margin: 0; color: rgba(255,255,255,0.85); font-size: 16px;'>{len(results_df)} candidates analyzed and ranked by AI</p>
            </div>
            <div style='background: linear-gradient(135deg, #06b6d4, #0891b2); padding: 16px 32px; border-radius: 50px;
                        font-weight: 700; font-size: 20px; box-shadow: 0 8px 24px rgba(0, 0, 0, 0.2);'>🏆 Top Matches</div>
        </div>
    </div>

    <div style='background: white; border-radius: 20px; overflow: hidden; box-shadow: 0 10px 40px rgba(0, 0, 0, 0.1); margin-bottom: 32px;'>
        <div style='overflow-x: auto;'>
        <table style='width: 100%; border-collapse: collapse; min-width: 800px;'>
            <thead>
                <tr style='background: linear-gradient(135deg, #1e1b4b, #312e81);'>
                    <th style='padding: 20px 24px; text-align: left; font-weight: 700; color: white; font-size: 14px; text-transform: uppercase; letter-spacing: 1px;'>Rank</th>
                    <th style='padding: 20px 24px; text-align: left; font-weight: 700; color: white; font-size: 14px; text-transform: uppercase; letter-spacing: 1px;'>Candidate</th>
                    <th style='padding: 20px 24px; text-align: center; font-weight: 700; color: white; font-size: 14px; text-transform: uppercase; letter-spacing: 1px;'>Overall</th>
                    <th style='padding: 20px 24px; text-align: center; font-weight: 700; color: white; font-size: 14px; text-transform: uppercase; letter-spacing: 1px;'>Semantic</th>
                    <th style='padding: 20px 24px; text-align: center; font-weight: 700; color: white; font-size: 14px; text-transform: uppercase; letter-spacing: 1px;'>Skills</th>
                    <th style='padding: 20px 24px; text-align: center; font-weight: 700; color: white; font-size: 14px; text-transform: uppercase; letter-spacing: 1px;'>Matched</th>
                </tr>
            </thead>
            <tbody>
    """

    for idx, row in results_df.iterrows():
        composite = row['composite_score'] * 100
        if composite >= 70:
            badge_gradient = "linear-gradient(135deg, #06b6d4, #0891b2)"
            rank_icon = "🥇" if idx == 0 else "🌟"
            row_bg = "#ecfeff"
        elif composite >= 50:
            badge_gradient = "linear-gradient(135deg, #f59e0b, #d97706)"
            rank_icon = "⚡"
            row_bg = "#fffbeb"
        else:
            badge_gradient = "linear-gradient(135deg, #ec4899, #be185d)"
            rank_icon = "💡"
            row_bg = "#fdf2f8"

        table_html += f"""
            <tr style='border-bottom: 1px solid #f3f4f6; background: {row_bg};'>
                <td style='padding: 20px 24px;'>
                    <div style='display: inline-flex; align-items: center; gap: 8px; background: {badge_gradient}; color: white;
                                padding: 8px 16px; border-radius: 50px; font-weight: 800; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.15);'>
                        <span style='font-size: 18px;'>{rank_icon}</span><span>#{idx + 1}</span>
                    </div>
                </td>
                <td style='padding: 20px 24px;'>
                    <div style='font-weight: 700; color: #1f2937; font-size: 16px; margin-bottom: 4px;'>{row['candidate_name']}</div>
                    <div style='font-size: 12px; color: #6b7280; text-transform: uppercase; letter-spacing: 0.5px;'>Profile ID</div>
                </td>
                <td style='padding: 20px 24px; text-align: center;'>
                    <div style='display: inline-block; background: {badge_gradient}; color: white; padding: 10px 20px; border-radius: 50px;
                               font-weight: 800; font-size: 18px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);'>{composite:.1f}%</div>
                </td>
                <td style='padding: 20px 24px; text-align: center;'>
                    <div style='display: inline-block; background: linear-gradient(135deg, #7c3aed, #a78bfa); color: white; padding: 8px 16px;
                               border-radius: 50px; font-weight: 700; font-size: 14px;'>{row['semantic_similarity'] * 100:.1f}%</div>
                </td>
                <td style='padding: 20px 24px; text-align: center;'>
                    <div style='display: inline-block; background: linear-gradient(135deg, #f59e0b, #fbbf24); color: white; padding: 8px 16px;
                               border-radius: 50px; font-weight: 700; font-size: 14px;'>{row['skill_match_percentage']:.1f}%</div>
                </td>
                <td style='padding: 20px 24px; text-align: center;'>
                    <div style='display: inline-block; background: linear-gradient(135deg, #ec4899, #f472b6); color: white; padding: 8px 16px;
                               border-radius: 50px; font-weight: 700; font-size: 14px;'>{row['matched_count']}/{row['total_required']}</div>
                </td>
            </tr>
        """

    table_html += """
            </tbody>
        </table>
        </div>
    </div>
    """

    detailed_html = """
    <div style='background: linear-gradient(to right, #1e1b4b 0%, #312e81 50%, #4c1d95 100%);
                padding: 40px; border-radius: 24px; color: white; margin-bottom: 32px;
                box-shadow: 0 20px 60px rgba(0, 0, 0, 0.3);'>
        <h1 style='margin: 0 0 8px 0; font-size: 36px; font-weight: 800;
                   background: linear-gradient(to right, #fff, #e0e7ff);
                   -webkit-background-clip: text; -webkit-text-fill-color: transparent;'>📋 Detailed Candidate Profiles</h1>
        <p style='margin: 0; color: rgba(255,255,255,0.85); font-size: 16px;'>In-depth analysis for each candidate</p>
    </div>
    """

    for idx, row in results_df.iterrows():
        detailed_html += f"<div style='margin-bottom: 40px;'>{format_single_result(row.to_dict())}</div>"

    export_df = results_df[
        ['candidate_name', 'composite_score', 'semantic_similarity', 'skill_match_percentage', 'matched_count',
         'total_required']].copy()
    export_df['composite_score'] = (export_df['composite_score'] * 100).round(2)
    export_df['semantic_similarity'] = (export_df['semantic_similarity'] * 100).round(2)

    return table_html + detailed_html, export_df


def process_single_resume(pdf_file, job_desc, progress=gr.Progress()):
    """Process single resume with progress tracking"""
    progress(0, desc="Starting analysis...")

    if pdf_file is None:
        return """<div style='background: linear-gradient(135deg, #ff6b6b, #c92a2a); padding: 32px; border-radius: 20px; color: white;
                    box-shadow: 0 8px 32px rgba(201, 42, 42, 0.2); display: flex; align-items: center; gap: 16px;'>
            <span style='font-size: 48px;'>📄</span><div><h3 style='margin: 0 0 8px 0; font-size: 24px; font-weight: 800;'>No Resume Uploaded</h3>
            <p style='margin: 0; opacity: 0.9;'>Please upload a PDF resume file to begin analysis</p></div></div>""", gr.update(visible=False)

    if not job_desc or job_desc.strip() == "":
        return """<div style='background: linear-gradient(135deg, #ff6b6b, #c92a2a); padding: 32px; border-radius: 20px; color: white;
                    box-shadow: 0 8px 32px rgba(201, 42, 42, 0.2); display: flex; align-items: center; gap: 16px;'>
            <span style='font-size: 48px;'>📝</span><div><h3 style='margin: 0 0 8px 0; font-size: 24px; font-weight: 800;'>No Job Description</h3>
            <p style='margin: 0; opacity: 0.9;'>Please enter a job description to match against</p></div></div>""", gr.update(visible=False)

    try:
        progress(0.2, desc="Extracting text from PDF...")
        result = system.screen_resume(pdf_file.name, job_desc)

        progress(0.8, desc="Generating report...")
        html_output = format_single_result(result)

        progress(0.9, desc="Creating export data...")
        df = pd.DataFrame([{
            'Candidate': result['candidate_name'],
            'Overall Match': f"{(result['semantic_similarity'] * 0.6 + result['skill_match_percentage'] / 100 * 0.4) * 100:.1f}%",
            'Semantic Match': f"{result['semantic_similarity'] * 100:.1f}%",
            'Skill Match': f"{result['skill_match_percentage']:.1f}%",
            'Matched Skills': ', '.join(result['matched_skills']),
            'Missing Skills': ', '.join(result['missing_skills'])
        }])

        progress(1.0, desc="Complete!")
        return html_output, gr.update(value=df, visible=True)
    except Exception as e:
        return f"""<div style='background: linear-gradient(135deg, #ff6b6b, #c92a2a); padding: 32px; border-radius: 20px; color: white;
                    box-shadow: 0 8px 32px rgba(201, 42, 42, 0.2); display: flex; align-items: center; gap: 16px;'>
            <span style='font-size: 48px;'>⚠️</span><div><h3 style='margin: 0 0 8px 0; font-size: 24px; font-weight: 800;'>Processing Error</h3>
            <p style='margin: 0; opacity: 0.9;'>{str(e)}</p></div></div>""", gr.update(visible=False)


def process_multiple_resumes(pdf_files, job_desc, progress=gr.Progress()):
    """Process multiple resumes with progress tracking"""
    progress(0, desc="Starting batch analysis...")

    if pdf_files is None or len(pdf_files) == 0:
        return """<div style='background: linear-gradient(135deg, #ff6b6b, #c92a2a); padding: 32px; border-radius: 20px; color: white;
                    box-shadow: 0 8px 32px rgba(201, 42, 42, 0.2); display: flex; align-items: center; gap: 16px;'>
            <span style='font-size: 48px;'>📚</span><div><h3 style='margin: 0 0 8px 0; font-size: 24px; font-weight: 800;'>No Resumes Uploaded</h3>
            <p style='margin: 0; opacity: 0.9;'>Please upload at least one PDF resume file</p></div></div>""", gr.update(visible=False)

    if not job_desc or job_desc.strip() == "":
        return """<div style='background: linear-gradient(135deg, #ff6b6b, #c92a2a); padding: 32px; border-radius: 20px; color: white;
                    box-shadow: 0 8px 32px rgba(201, 42, 42, 0.2); display: flex; align-items: center; gap: 16px;'>
            <span style='font-size: 48px;'>📝</span><div><h3 style='margin: 0 0 8px 0; font-size: 24px; font-weight: 800;'>No Job Description</h3>
            <p style='margin: 0; opacity: 0.9;'>Please enter a job description to match against</p></div></div>""", gr.update(visible=False)

    try:
        results = []
        total_files = len(pdf_files)

        for idx, pdf_file in enumerate(pdf_files):
            progress((idx / total_files) * 0.8, desc=f"Processing resume {idx + 1}/{total_files}...")
            result = system.screen_resume(pdf_file.name, job_desc)
            if 'error' not in result:
                results.append(result)

        if not results:
            return """<div style='background: linear-gradient(135deg, #ff6b6b, #c92a2a); padding: 32px; border-radius: 20px; color: white;
                        box-shadow: 0 8px 32px rgba(201, 42, 42, 0.2); display: flex; align-items: center; gap: 16px;'>
                <span style='font-size: 48px;'>❌</span><div><h3 style='margin: 0 0 8px 0; font-size: 24px; font-weight: 800;'>Processing Failed</h3>
                <p style='margin: 0; opacity: 0.9;'>No valid resumes could be processed</p></div></div>""", gr.update(visible=False)

        progress(0.85, desc="Ranking candidates...")
        df = pd.DataFrame(results)
        df['composite_score'] = (df['semantic_similarity'] * 0.6 + df['skill_match_percentage'] / 100 * 0.4)
        df = df.sort_values('composite_score', ascending=False).reset_index(drop=True)

        progress(0.95, desc="Generating reports...")
        html_output, export_df = format_multiple_results(df)

        progress(1.0, desc="Complete!")
        return html_output, gr.update(value=export_df, visible=True)
    except Exception as e:
        return f"""<div style='background: linear-gradient(135deg, #ff6b6b, #c92a2a); padding: 32px; border-radius: 20px; color: white;
                    box-shadow: 0 8px 32px rgba(201, 42, 42, 0.2); display: flex; align-items: center; gap: 16px;'>
            <span style='font-size: 48px;'>⚠️</span><div><h3 style='margin: 0 0 8px 0; font-size: 24px; font-weight: 800;'>Processing Error</h3>
            <p style='margin: 0; opacity: 0.9;'>{str(e)}</p></div></div>""", gr.update(visible=False)


# Gradio Interface
with gr.Blocks(
        theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="purple", neutral_hue="slate"),
        css="""
        .gradio-container { background: #ffffff !important; font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 100% !important; }
        .progress-bar { display: block !important; visibility: visible !important; }
        .main-header { text-align: center; padding: 32px 24px; background: linear-gradient(135deg, #667eea 0%, #764ba2 50%, #f093fb 100%);
                      color: white; border-radius: 20px; margin-bottom: 32px; box-shadow: 0 8px 32px rgba(102, 126, 234, 0.3); }
        .tab-nav button { font-weight: 600 !important; font-size: 16px !important; border-radius: 12px 12px 0 0 !important;
                         padding: 16px 32px !important; transition: all 0.3s ease !important; }
        .tab-nav button[aria-selected="true"] { background: linear-gradient(135deg, #667eea, #764ba2) !important;
                                                color: white !important; box-shadow: 0 -4px 16px rgba(102, 126, 234, 0.3) !important; }
        .upload-area { border: 3px dashed rgba(102, 126, 234, 0.5) !important; border-radius: 16px !important;
                      background: #fafafa !important; transition: all 0.3s ease !important; }
        .upload-area:hover { border-color: #667eea !important; background: #f5f3ff !important; }
        button.primary { background: linear-gradient(135deg, #667eea, #764ba2) !important; border: none !important;
                        font-weight: 700 !important; font-size: 18px !important; padding: 16px 48px !important;
                        border-radius: 12px !important; box-shadow: 0 8px 24px rgba(102, 126, 234, 0.4) !important;
                        transition: all 0.3s ease !important; }
        button.primary:hover { transform: translateY(-2px) !important; box-shadow: 0 12px 32px rgba(102, 126, 234, 0.5) !important; }
        textarea { border-radius: 12px !important; border: 2px solid #e5e7eb !important; background: #fafafa !important;
                  transition: all 0.3s ease !important; }
        textarea:focus { border-color: #667eea !important; box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.1) !important; background: #ffffff !important; }
        .progress-container { margin: 20px 0; }
    """
) as demo:
    gr.HTML("""
        <div class="main-header">
            <div style='display: inline-block; background: rgba(255, 255, 255, 0.2); padding: 6px 18px; border-radius: 50px; margin-bottom: 12px;
                        backdrop-filter: blur(10px); border: 1px solid rgba(255, 255, 255, 0.3);'>
                <span style='font-size: 12px; font-weight: 700; text-transform: uppercase; letter-spacing: 2px;'>✨ Next-Gen Recruitment</span>
            </div>
            <h1 style="margin: 0 0 8px 0; font-size: 38px; font-weight: 900; text-shadow: 0 2px 10px rgba(0, 0, 0, 0.2);">HireVision </h1>
            <p style="margin: 0; font-size: 16px; opacity: 0.95; font-weight: 500;">Your AI-powered hiring assistant</p>
        </div>
    """)

    with gr.Tabs():
        with gr.Tab("📄 Single Resume Analysis"):
            gr.Markdown("### 🎯 Individual Candidate Deep Dive\nUpload a single resume for comprehensive AI analysis and matching")
            with gr.Row():
                with gr.Column(scale=1):
                    single_pdf = gr.File(label="📎 Upload Resume", file_types=[".pdf"], elem_classes=["upload-area"])
                    single_jd = gr.Textbox(
                        label="💼 Job Description",
                        placeholder="Paste the complete job description here...",
                        lines=15
                    )
                    single_btn = gr.Button("🔍 Analyze Resume", variant="primary", size="lg", elem_classes=["primary"])
                with gr.Column(scale=2):
                    single_output = gr.HTML(label="📊 Analysis Results")
                    single_export = gr.Dataframe(label="📥 Export Data (CSV)", visible=False, interactive=False)

            single_btn.click(
                fn=process_single_resume,
                inputs=[single_pdf, single_jd],
                outputs=[single_output, single_export]
            )

        with gr.Tab("📚 Batch Resume Ranking"):
            gr.Markdown("### 🏆 Multi-Candidate Comparison\nUpload multiple resumes to get AI-powered ranking and side-by-side comparison")
            with gr.Row():
                with gr.Column(scale=1):
                    multiple_pdfs = gr.File(
                        label="📎 Upload Multiple Resumes",
                        file_count="multiple",
                        file_types=[".pdf"],
                        elem_classes=["upload-area"]
                    )
                    multiple_jd = gr.Textbox(
                        label="💼 Job Description",
                        placeholder="Paste the complete job description here...",
                        lines=15
                    )
                    multiple_btn = gr.Button("🔍 Analyze & Rank All", variant="primary", size="lg", elem_classes=["primary"])
                with gr.Column(scale=2):
                    multiple_output = gr.HTML(label="📊 Rankings & Analysis")
                    multiple_export = gr.Dataframe(label="📥 Export Rankings (CSV)", visible=False, interactive=False)

            multiple_btn.click(
                fn=process_multiple_resumes,
                inputs=[multiple_pdfs, multiple_jd],
                outputs=[multiple_output, multiple_export]
            )

if __name__ == "__main__":
    print("Launching Gradio interface...")
    demo.launch(share=True)

Initializing Resume Screening System...
loading full_matcher ...
loading abv_matcher ...
loading full_uni_matcher ...
loading low_form_matcher ...
loading token_matcher ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


System Ready!

Launching Gradio interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f4364830148c273b56.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
